##### Import libraries

In [2]:
import IPython
import pandas
import sparqldataframe
from SPARQLWrapper import SPARQLWrapper, JSON, CSV
import os 
import subprocess
import time
import pandas as pd 

##### Useful functions

In [3]:
def displaySparqlResults(results):
    '''
    Displays as HTML the result of a SPARQLWrapper query in a Jupyter notebook.
    
        Parameters:
            results (dictionnary): the result of a call to SPARQLWrapper.query().convert()
    '''
    variableNames = results['head']['vars']
    tableCode = '<table><tr><th>{}</th></tr><tr>{}</tr></table>'.format('</th><th>'.join(variableNames), '</tr><tr>'.join('<td>{}</td>'.format('</td><td>'.join([row[vName]['value'] if vName in row.keys() else "&nbsp;" for vName in variableNames]))for row in results["results"]["bindings"]))
    IPython.display.display(IPython.display.HTML(tableCode))

##### Define file paths and prefixes

In [4]:
reactomeVersion = 94
prefixes = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:<http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX dcterms: <http://purl.org/dc/terms/>

PREFIX chebi: <http://purl.obolibrary.org/obo/chebi/>
PREFIX chebidb: <http://purl.obolibrary.org/obo/CHEBI_>
PREFIX chebirel: <http://purl.obolibrary.org/obo/CHEBI#>
PREFIX oboInOwl: <http://www.geneontology.org/formats/oboInOwl#>

PREFIX bp3: <http://www.biopax.org/release/biopax-level3.owl#>

PREFIX reactome: <http://www.reactome.org/biopax/{}#>
""".format(reactomeVersion)

biopaxURI = "http://www.biopax.org/release/biopax-level3.owl#"
reactomeURI = "http://www.reactome.org/biopax/{}#".format(reactomeVersion)
uniprotURI = "http://purl.uniprot.org/uniprot/"

current_directory = os.getcwd()
endpoint_reactome = "http://localhost:3030/reactome/query"
rdfFormat = "turtle"
BioPAX_Ontology_file_path = os.path.join(current_directory, '../', 'Data', 'biopax_ontology/biopax-level3.owl')
ReactomeBioPAX_file_path = os.path.join(current_directory, '../', 'Data', 'reactome/Homo_sapiens_v94.owl')

##### Launch SPARQL endpoint

In [5]:
command = [
    '/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0/fuseki-server',
    '--file', ReactomeBioPAX_file_path,
    '--file', BioPAX_Ontology_file_path,
    '/reactome']

process = subprocess.Popen(command)
time.sleep(60)

19:35:32 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2025/BioPAX-To-SIF-SPARQL/Scripts/../Data/reactome/Homo_sapiens_v94.owl
19:35:33 WARN  riot            :: [line: 66845, col: 48] {W137} Input is large. Switching off checking for illegal reuse of rdf:ID's.
19:35:54 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2025/BioPAX-To-SIF-SPARQL/Scripts/../Data/biopax_ontology/biopax-level3.owl
19:35:54 INFO  Server          :: Running in read-only mode for /reactome
19:35:54 INFO  Server          :: Apache Jena Fuseki 4.9.0
19:35:54 INFO  Config          :: FUSEKI_HOME=/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0
19:35:54 INFO  Config          :: FUSEKI_BASE=/home/cbeust/Projects/2025/BioPAX-To-SIF-SPARQL/Scripts/run
19:35:54 INFO  Config          :: Shiro file: file:///home/cbeust/Projects/2025/BioPAX-To-SIF-SPARQL/Scripts/run/shiro.ini
19:35:54 INFO  Server          :: Database: in-memory, with files loaded
19:3

##### Query the BioPAX export of Reactome to get the list of all ProteinReferences and SmallMoleculeReferences

In [7]:
query="""
SELECT DISTINCT ?entityRef ?entityRefName ?entityID
WHERE {
  VALUES ?entityRefType { bp3:ProteinReference bp3:SmallMoleculeReference }
  ?entityRef rdf:type ?entityRefType .
  ?entityRef bp3:xref ?entityRefXref .
  ?entityRefXref bp3:id ?entityID .
  ?entityRef bp3:name ?entityRefName .
}
"""
sparql = SPARQLWrapper(endpoint_reactome)
sparql.setQuery(prefixes+query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
#displaySparqlResults(results)

sparql.setReturnFormat(CSV)
results = sparql.query().convert()
with open(f"../Results/ReactomeHomoSapiens94/UtilityFiles/ReactomeHomoSapiens94EntityRefs.csv", "wb") as f:
    f.write(results)

19:37:09 INFO  Fuseki          :: [3] GET http://localhost:3030/reactome/query?query=%0APREFIX+rdf%3A+%3Chttp%3A//www.w3.org/1999/02/22-rdf-syntax-ns%23%3E%0APREFIX+rdfs%3A%3Chttp%3A//www.w3.org/2000/01/rdf-schema%23%3E%0APREFIX+owl%3A+%3Chttp%3A//www.w3.org/2002/07/owl%23%3E%0APREFIX+xsd%3A+%3Chttp%3A//www.w3.org/2001/XMLSchema%23%3E%0APREFIX+dc%3A+%3Chttp%3A//purl.org/dc/elements/1.1/%3E%0APREFIX+dcterms%3A+%3Chttp%3A//purl.org/dc/terms/%3E%0A%0APREFIX+chebi%3A+%3Chttp%3A//purl.obolibrary.org/obo/chebi/%3E%0APREFIX+chebidb%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI_%3E%0APREFIX+chebirel%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI%23%3E%0APREFIX+oboInOwl%3A+%3Chttp%3A//www.geneontology.org/formats/oboInOwl%23%3E%0A%0APREFIX+bp3%3A+%3Chttp%3A//www.biopax.org/release/biopax-level3.owl%23%3E%0A%0APREFIX+reactome%3A+%3Chttp%3A//www.reactome.org/biopax/94%23%3E%0A%0ASELECT+DISTINCT+%3FentityRef+%3FentityRefName+%3FentityID%0AWHERE+%7B%0A++VALUES+%3FentityRefType+%7B+bp3%3AProteinReferen

##### Generate node table of SIF abstraction of Reactome

In [8]:
SIFabstraction = pd.read_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94.csv", sep=",", header=None)
print(SIFabstraction.head())

NodeTableSIF = pd.DataFrame(columns=['Node', 'Type', 'EntityName'])
SIFentities = list()

for index, row in SIFabstraction.iterrows():
    if not row[0] in SIFentities:
        SIFentities.append(row[0])
    if not row[2] in SIFentities:
        SIFentities.append(row[2])

print(len(SIFentities))

NodeTableSIF['Node'] = SIFentities
NodeTypes = list()
for index, row in NodeTableSIF.iterrows():
    if "Protein" in row[0]:
        NodeTypes.append("Protein")
    elif "SmallMolecule" in row[0]:
        NodeTypes.append("SmallMolecule")
NodeTableSIF['Type'] = NodeTypes

print(NodeTableSIF)

                               0                                  1  \
0  reactome:ProteinReference4833  abstraction:ControlsStateChangeOf   
1  reactome:ProteinReference4452  abstraction:ControlsStateChangeOf   
2  reactome:ProteinReference3670  abstraction:ControlsStateChangeOf   
3  reactome:ProteinReference5974  abstraction:ControlsStateChangeOf   
4  reactome:ProteinReference3034  abstraction:ControlsStateChangeOf   

                               2  
0  reactome:ProteinReference7186  
1  reactome:ProteinReference4453  
2  reactome:ProteinReference3669  
3  reactome:ProteinReference6773  
4  reactome:ProteinReference3085  
8516
                                     Node           Type EntityName
0           reactome:ProteinReference4833        Protein        NaN
1           reactome:ProteinReference7186        Protein        NaN
2           reactome:ProteinReference4452        Protein        NaN
3           reactome:ProteinReference4453        Protein        NaN
4           reacto

/tmp/ipykernel_20791/3519871853.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if "Protein" in row[0]:
/tmp/ipykernel_20791/3519871853.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  elif "SmallMolecule" in row[0]:


In [9]:
PathwayEntities = pd.read_csv("../Results/ReactomeHomoSapiens94/UtilityFiles/ReactomeHomoSapiens94EntityRefs.csv", sep=",", header=0)

dicoEntityNames = dict()
dicoEntityIDs = dict()
for item,row in PathwayEntities.iterrows():
    entityURI = f"reactome:{row[0][40:]}"
    dicoEntityNames[entityURI] = row[1]
    dicoEntityIDs[entityURI] = row[2]

SIFentityNames = list()
SIFentityIDs = list()
for index, row in NodeTableSIF.iterrows():
    entity = row[0]
    SIFentityNames.append(dicoEntityNames[entity])
    SIFentityIDs.append(dicoEntityIDs[entity])

NodeTableSIF['EntityName'] = SIFentityNames
NodeTableSIF['EntityID'] = SIFentityIDs
print(NodeTableSIF)
print(NodeTableSIF.head())

/tmp/ipykernel_20791/1449288390.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  entityURI = f"reactome:{row[0][40:]}"
/tmp/ipykernel_20791/1449288390.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dicoEntityNames[entityURI] = row[1]
/tmp/ipykernel_20791/1449288390.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dicoEntityIDs[entityURI] = row[2]


                                     Node           Type  \
0           reactome:ProteinReference4833        Protein   
1           reactome:ProteinReference7186        Protein   
2           reactome:ProteinReference4452        Protein   
3           reactome:ProteinReference4453        Protein   
4           reactome:ProteinReference3670        Protein   
...                                   ...            ...   
8511  reactome:SmallMoleculeReference2864  SmallMolecule   
8512  reactome:SmallMoleculeReference1878  SmallMolecule   
8513  reactome:SmallMoleculeReference1866  SmallMolecule   
8514  reactome:SmallMoleculeReference2359  SmallMolecule   
8515  reactome:SmallMoleculeReference2383  SmallMolecule   

                                             EntityName      EntityID  
0                                  UniProt:P16035 TIMP2        P16035  
1                                 UniProt:Q03167 TGFBR3        Q03167  
2                                   UniProt:P27487 DPP4        

/tmp/ipykernel_20791/1449288390.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  entity = row[0]


In [10]:
NodeTableSIF.to_csv("../Results/ReactomeHomoSapiens94/UtilityFiles/NodeTableSIF_ReactomeHomoSapiens94.csv", sep=",", index=False)

19:59:21 INFO  Fuseki          :: [8] POST http://localhost:3030/reactome/sparql
19:59:21 INFO  Fuseki          :: [8] Query = PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> PREFIX rdfs:<http://www.w3.org/2000/01/rdf-schema#> PREFIX owl: <http://www.w3.org/2002/07/owl#> PREFIX xsd: <http://www.w3.org/2001/XMLSchema#> PREFIX dc: <http://purl.org/dc/elements/1.1/> PREFIX dcterms: <http://purl.org/dc/terms/> PREFIX chebi: <http://purl.obolibrary.org/obo/chebi/> PREFIX chebidb: <http://purl.obolibrary.org/obo/CHEBI_> PREFIX chebirel: <http://purl.obolibrary.org/obo/CHEBI#> PREFIX oboInOwl: <http://www.geneontology.org/formats/oboInOwl#> PREFIX bp3: <http://www.biopax.org/release/biopax-level3.owl#> PREFIX reactome: <http://www.reactome.org/biopax/93/48887#> PREFIX abstraction:<http://abstraction/#>  SELECT ?id WHERE {    VALUES ?entity { <http://www.reactome.org/biopax/40/48887#ProteinReference7280>  }   ?entity bp3:id ?id . } 
19:59:21 INFO  Fuseki          :: [8] 200 OK (16 ms

In [11]:
process.kill()
time.sleep(60)